# Wild Animal Species: ImageNet Inference with the Multi-Weights API

A **wildlife research platform** wants to automatically tag photographs of animals using a pretrained deep learning model. Instead of training from scratch, load a stock EfficientNet_B0 through torchvision's Multi-Weights API, apply the preprocessing bundle that ships with the weights, and confirm that the model correctly identifies each animal species by its ImageNet class name.

## Tasks

1. Load `EfficientNet_B0_Weights.DEFAULT` as the weights object and build the EfficientNet_B0 model using the `weights=` keyword argument.

2. Obtain the preprocessing transform by calling `weights.transforms()` on the loaded weights object.

3. For each image in the dataset, open it with PIL, convert to RGB, apply the transform, and add a batch dimension so the tensor has shape `(1, 3, H, W)`.

4. Put the model in evaluation mode and run inference inside a `torch.no_grad()` block.

5. Retrieve ImageNet class names from `weights.meta["categories"]` and map the highest-scoring output index to a predicted class name. Store all predictions in a variable called `predictions` as a list of strings.

6. Inspect `model.classifier` to read the `in_features` and `out_features` of its `Linear` layer. Store these integers in variables called `classifier_in_features` and `classifier_out_features`.

In [ ]:
# Run this cell before writing your solution
import warnings
warnings.filterwarnings('ignore')

import os
import zipfile
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image

# Extract student images into the images/ folder
zipfile.ZipFile("images_TRNSF1.zip").extractall("images")

print("Archive extracted. Images available under: images/animals/animals/")
for cls in sorted(os.listdir(os.path.join("images", "animals", "animals"))):
    folder = os.path.join("images", "animals", "animals", cls)
    if os.path.isdir(folder):
        count = len([f for f in os.listdir(folder)])
        print(f"  {cls}: {count} images")

In [ ]:
# Task 1: Load EfficientNet_B0_Weights.DEFAULT and build the model
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Task 2: Obtain the preprocessing transform from the weights object
transform = weights.transforms()

# Task 4: Put the model in evaluation mode
model.eval()

# Task 5: Run inference on all student images and collect predictions
categories = weights.meta["categories"]

IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.ppm', '.pgm', '.tif', '.tiff', '.gif')

predictions = []
image_root = os.path.join("images", "animals", "animals")

for class_name in sorted(os.listdir(image_root)):
    class_dir = os.path.join(image_root, class_name)
    if not os.path.isdir(class_dir):
        continue
    for fname in sorted(os.listdir(class_dir)):
        if fname.lower().endswith(IMAGE_EXTS):
            fpath = os.path.join(class_dir, fname)
            # Task 3: Open each image, convert to RGB, apply transform, add batch dimension
            img = Image.open(fpath).convert("RGB")
            tensor = transform(img).unsqueeze(0)
            # Task 4: Run inference inside torch.no_grad()
            with torch.no_grad():
                output = model(tensor)
            pred_idx = output.argmax(dim=1).item()
            predictions.append(categories[pred_idx])

# Task 6: Read classifier in_features and out_features from the Linear layer
for layer in model.classifier:
    if isinstance(layer, nn.Linear):
        classifier_in_features = layer.in_features
        classifier_out_features = layer.out_features
        break

print(f"Total predictions: {len(predictions)}")
print(f"Sample predictions: {predictions[:5]}")
print(f"Unique predicted classes: {len(set(predictions))}")
print(f"classifier_in_features: {classifier_in_features}")
print(f"classifier_out_features: {classifier_out_features}")